# 7. STATLOG AUSTRALIAN CREDIT - Preprocesamiento
**Tipo:** CLASIFICACIÓN BINARIA (aprobar/rechazar crédito)
**Variable objetivo:** A15 (1 = Aprobado, 0 = Rechazado)
**NOTA:** Esta versión está PRE-LIMPIA (sin nulos)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# CARGAR DATOS (.dat separado por espacios)
# ============================================================
columnas = ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 
            'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15']

df = pd.read_csv('australian.dat', sep=' ', names=columnas)
print(f"Dimensiones: {df.shape}")
print(f"\nNulos: {df.isnull().sum().sum()}")  # Debería ser 0
df.head()

In [ ]:
# ============================================================
# SEPARAR VARIABLE OBJETIVO
# ============================================================
y = df['A15'].values
X_df = df.drop(columns=['A15'])

print(f"Balance de clases: {np.bincount(y)}")
print(f"Proporción aprobados: {y.mean():.2%}")

In [ ]:
# ============================================================
# IDENTIFICAR TIPOS DE VARIABLES
# ============================================================
# Según documentación Statlog:
# Continuas: A2, A3, A7, A10, A13, A14
# Categóricas (aunque son números): A1, A4, A5, A6, A8, A9, A11, A12

continuas = ['A2', 'A3', 'A7', 'A10', 'A13', 'A14']
categoricas = ['A4', 'A5', 'A6', 'A12']  # Con múltiples valores
binarias = ['A1', 'A8', 'A9', 'A11']  # Ya son 0/1

print(f"Continuas: {continuas}")
print(f"Categóricas (necesitan One-Hot): {categoricas}")
print(f"Binarias (listas): {binarias}")

In [ ]:
# ============================================================
# ONE-HOT ENCODING PARA CATEGÓRICAS
# ============================================================
# A4 tiene 3 valores, A5 tiene 14, A6 tiene 9, A12 tiene 3
# IMPORTANTE: Aunque son números, NO tienen orden matemático

X_encoded = pd.get_dummies(X_df, columns=categoricas, drop_first=True)
print(f"Dimensiones después de encoding: {X_encoded.shape}")
print(f"Columnas: {list(X_encoded.columns)}")

In [ ]:
# ============================================================
# DIVISIÓN DE DATOS
# ============================================================
X = X_encoded.to_numpy()

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.333, random_state=42)

print(f"Train: {X_train.shape[0]}")
print(f"Val: {X_val.shape[0]}")
print(f"Test: {X_test.shape[0]}")

In [ ]:
# ============================================================
# NORMALIZACIÓN
# ============================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("✅ Datos listos para CLASIFICACIÓN BINARIA")
print(f"Características: {X_train.shape[1]}")

## MÉTODO TRADICIONAL

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

## CON PYTORCH

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).reshape(-1, 1)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

class ClasificadorCredito(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.red(x)

modelo = ClasificadorCredito(X_train_t.shape[1])
criterio = nn.BCELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)
print(modelo)